In [ ]:
import os
import pathlib
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import matplotlib.pyplot as plt

In [ ]:
DATASET_PATH = tf.keras.utils.get_file(
    'speech_commands_v0.02',
    origin="http://storage.googleapis.com/download.tensorflow.org/data/speech_commands_v0.02.tar.gz",
    extract=True,
    cache_dir='.', cache_subdir='data'
)

DATASET_PATH = pathlib.Path(DATASET_PATH)

2428923189/2428923189 ━━━━━━━━━━━━━━━━━━━━ 13s 0us/step


In [ ]:
commands = np.array(tf.io.gfile.listdir(str(DATASET_PATH)))
commands = commands[commands != 'README.md']
print("Commands:", commands)

Commands: ['bed' 'forward' 'sheila' 'wow' 'LICENSE' 'off' 'six' 'happy' 'up' 'eight'
 'nine' '.DS_Store' 'testing_list.txt' 'yes' 'house' 'follow' 'visual'
 'seven' 'two' 'bird' '_background_noise_' 'three' 'one' 'on' 'cat'
 'learn' 'stop' 'zero' 'go' 'dog' 'down' 'backward' 'four' 'right' 'left'
 'validation_list.txt' 'five' 'tree' 'marvin' 'no']


In [ ]:
train_ds, val_ds = tf.keras.utils.audio_dataset_from_directory(
    DATASET_PATH,
    batch_size=32,
    validation_split=0.2,
    output_sequence_length=16000,
    seed=42,
    subset='both'
)

Found 105835 files belonging to 36 classes.
Using 84668 files for training.
Using 21167 files for validation.


In [ ]:
def get_spectrogram(waveform):
    spectrogram = tf.signal.stft(
        waveform,
        frame_length=255,
        frame_step=128
    )
    spectrogram = tf.abs(spectrogram)
    return spectrogram

In [ ]:
def preprocess(audio, label):
    audio = tf.squeeze(audio, axis=-1)
    spectrogram = get_spectrogram(audio)
    return spectrogram, label

train_ds = train_ds.map(preprocess)
val_ds = val_ds.map(preprocess)

In [ ]:
for spec, label in train_ds.take(1):
    input_shape = spec.shape[1:]
print("Input shape:", input_shape)

Input shape: (124, 129)


In [ ]:
model = tf.keras.Sequential([
    layers.Input(shape=input_shape),

    layers.Reshape((input_shape[0], input_shape[1])),

    layers.LSTM(128, return_sequences=False),

    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),

    layers.Dense(len(commands), activation='softmax')
])

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

Epoch 1/10
2646/2646 ━━━━━━━━━━━━━━━━━━━━ 586s 220ms/step - accuracy: 0.1033 - loss: 3.1777 - val_accuracy: 0.2755 - val_loss: 2.4699
Epoch 2/10
2646/2646 ━━━━━━━━━━━━━━━━━━━━ 615s 217ms/step - accuracy: 0.3314 - loss: 2.2654 - val_accuracy: 0.5816 - val_loss: 1.4258
Epoch 3/10
2646/2646 ━━━━━━━━━━━━━━━━━━━━ 569s 215ms/step - accuracy: 0.6041 - loss: 1.3701 - val_accuracy: 0.7386 - val_loss: 0.8788
Epoch 4/10
2646/2646 ━━━━━━━━━━━━━━━━━━━━ 578s 219ms/step - accuracy: 0.7341 - loss: 0.9242 - val_accuracy: 0.7851 - val_loss: 0.7276
Epoch 5/10
2646/2646 ━━━━━━━━━━━━━━━━━━━━ 587s 222ms/step - accuracy: 0.7842 - loss: 0.7507 - val_accuracy: 0.8123 - val_loss: 0.6539
Epoch 6/10
2646/2646 ━━━━━━━━━━━━━━━━━━━━ 611s 231ms/step - accuracy: 0.8167 - loss: 0.6394 - val_accuracy: 0.8344 - val_loss: 0.5649
Epoch 7/10
2646/2646 ━━━━━━━━━━━━━━━━━━━━ 595s 221ms/step - accuracy: 0.8367 - loss: 0.5636 - val_accuracy: 0.8514 - val_loss: 0.5130
Epoch 8/10
2646/2646 ━━━━━━━━━━━━━━━━━━━━ 626s 222ms/step - ac

In [ ]:
test_loss, test_acc = model.evaluate(val_ds)
print("Validation Accuracy:", test_acc)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 470ms/step
Predicted: bird
Actual: bird


In [ ]:
from IPython.display import Audio

In [ ]:
for spec, label in val_ds.take(1):
    waveform = tf.squeeze(spec, axis=-1)

    prediction = model.predict(spec)

    predicted_index = np.argmax(prediction[0])
    true_index = label.numpy()[0]

    print("Predicted:", commands[predicted_index])
    print("Actual:", commands[true_index])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 169ms/step
Predicted: bird
Actual: bird


In [ ]:
for spec, label in val_ds.take(1):

    prediction = model.predict(spec)

    for i in range(len(spec)):

        pred_index = np.argmax(prediction[i])
        true_index = label.numpy()[i]

        print(f"Sample {i+1}")
        print("Predicted:", commands[pred_index])
        print("Actual:", commands[true_index])
        print("----")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 169ms/step
Sample 1
Predicted: bird
Actual: bird
----
Sample 2
Predicted: bird
Actual: bird
----
Sample 3
Predicted: zero
Actual: zero
----
Sample 4
Predicted: visual
Actual: visual
----
Sample 5
Predicted: yes
Actual: yes
----
Sample 6
Predicted: cat
Actual: cat
----
Sample 7
Predicted: forward
Actual: forward
----
Sample 8
Predicted: off
Actual: off
----
Sample 9
Predicted: visual
Actual: visual
----
Sample 10
Predicted: LICENSE
Actual: LICENSE
----
Sample 11
Predicted: cat
Actual: cat
----
Sample 12
Predicted: backward
Actual: backward
----
Sample 13
Predicted: backward
Actual: backward
----
Sample 14
Predicted: yes
Actual: yes
----
Sample 15
Predicted: left
Actual: left
----
Sample 16
Predicted: right
Actual: right
----
Sample 17
Predicted: happy
Actual: sheila
----
Sample 18
Predicted: three
Actual: three
----
Sample 19
Predicted: one
Actual: one
----
Sample 20
Predicted: up
Actual: up
----
Sample 21
Predicted: visual
Actual: visual
----
Sample 22
Predi

In [ ]:
raw_train_ds, raw_val_ds = tf.keras.utils.audio_dataset_from_directory(
    DATASET_PATH,
    batch_size=1,
    validation_split=0.2,
    seed=42,
    output_sequence_length=16000,
    subset='both'
)

Found 105835 files belonging to 36 classes.
Using 84668 files for training.
Using 21167 files for validation.


In [ ]:
commands = raw_val_ds.class_names

In [ ]:
from IPython.display import Audio
import numpy as np

for audio, label in raw_val_ds.take(1):

    # Remove last channel dimension
    waveform = tf.squeeze(audio, axis=-1)

    # Convert to spectrogram
    spec = tf.signal.stft(
        waveform,
        frame_length=255,
        frame_step=128
    )
    spec = tf.abs(spec)

    # Predict
    prediction = model.predict(spec)
    pred_labels = np.argmax(prediction, axis=1)

    for i in range(len(waveform)):

        print("Sample:", i+1)
        print("Predicted:", commands[pred_labels[i]])
        print("Actual:", commands[label.numpy()[i]])

        # 🔊 Play audio
        display(Audio(waveform[i].numpy(), rate=16000))

        print("------------")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step
Sample: 1
Predicted: no
Actual: no


------------


In [ ]:

for audio, label in raw_val_ds.take(10):
    waveform = tf.squeeze(audio, axis=-1)
    # Play
    display(Audio(waveform.numpy(), rate=16000))

    # Convert to spectrogram
    spec = get_spectrogram(waveform)


    # Predict
    prediction = model.predict(spec)
    print("Predicted:", commands[np.argmax(prediction)])
Audio(waveform.numpy(), rate=16000)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
Predicted: no


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
Predicted: no


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
Predicted: stop


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
Predicted: left


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
Predicted: happy


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
Predicted: seven


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
Predicted: backward


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
Predicted: dog


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
Predicted: left


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
Predicted: cat
